# Notebook 3 — Backtesting + Claude API Signal Layer
## PS6: Chart Pattern Intelligence | ET AI Hackathon 2026

**What this notebook does:**
- Backtests all detected patterns: forward returns at +5, +10, +20 days
- Computes win rate, avg gain, avg loss, profit factor per pattern type per stock
- Calls Claude API to generate plain-English signals from pattern + stats
- Caches all Claude outputs to avoid repeated API calls

**Requires:** `ANTHROPIC_API_KEY` environment variable set.

In [1]:
# Imports
import pandas as pd
import numpy as np
import anthropic
import json
import os
import time
import hashlib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('ps6_data')
CACHE_FILE = BASE_DIR / 'claude_cache.json'
BACKTEST_FILE = BASE_DIR / 'backtest_results.parquet'

# Set your API key here or via environment variable
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY automatically
print('Anthropic client ready.')

Anthropic client ready.


In [2]:
# Step 1: Backtesting engine

class PatternBacktester:
    """
    For each detected pattern, compute forward returns and win statistics.
    Forward periods: 5 days, 10 days, 20 days.
    """

    FORWARD_PERIODS = [5, 10, 20]

    def __init__(self, df: pd.DataFrame, symbol: str):
        self.df = df.reset_index()
        self.symbol = symbol

    def compute_forward_return(self, entry_idx: int, period: int) -> float:
        """Return % change from entry_idx to entry_idx + period."""
        if entry_idx + period >= len(self.df):
            return np.nan
        entry_price = self.df['close'].iloc[entry_idx]
        exit_price  = self.df['close'].iloc[entry_idx + period]
        return (exit_price - entry_price) / entry_price * 100

    def backtest_detection(self, detection: dict) -> dict:
        """Compute stats for one detected pattern instance."""
        end_idx = detection.get('end_idx', 0)
        result = detection.copy()
        for p in self.FORWARD_PERIODS:
            fwd = self.compute_forward_return(end_idx, p)
            result[f'fwd_{p}d_return'] = round(fwd, 3) if not np.isnan(fwd) else None
            result[f'fwd_{p}d_win'] = fwd > 0 if not np.isnan(fwd) else None
        return result

    @staticmethod
    def compute_win_stats(detections: list) -> dict:
        """Aggregate win rate, avg gain/loss across a list of detections."""
        if not detections:
            return {}
        df = pd.DataFrame(detections)
        stats = {}
        for p in PatternBacktester.FORWARD_PERIODS:
            col_ret = f'fwd_{p}d_return'
            col_win = f'fwd_{p}d_win'
            if col_ret not in df.columns:
                continue
            valid = df[col_ret].dropna()
            if len(valid) == 0:
                continue
            wins = valid[valid > 0]
            losses = valid[valid <= 0]
            stats[f'{p}d_win_rate']   = round(len(wins) / len(valid) * 100, 1)
            stats[f'{p}d_avg_gain']   = round(wins.mean(), 2) if len(wins) > 0 else 0
            stats[f'{p}d_avg_loss']   = round(losses.mean(), 2) if len(losses) > 0 else 0
            stats[f'{p}d_profit_factor'] = round(
                abs(wins.sum() / losses.sum()), 2) if losses.sum() != 0 else np.inf
            stats[f'{p}d_sample_size'] = len(valid)
        return stats


# Run backtest on all stored detections
def run_full_backtest(ohlcv_dir: Path, detections_file: Path) -> pd.DataFrame:
    if not detections_file.exists():
        print('Run Notebook 2 first to generate detections.')
        return pd.DataFrame()

    df_det = pd.read_parquet(detections_file)
    all_results = []

    for symbol, grp in df_det.groupby('symbol'):
        safe = symbol.replace('.', '_')
        pf = ohlcv_dir / f'{safe}.parquet'
        if not pf.exists():
            continue
        df_stock = pd.read_parquet(pf)
        bt = PatternBacktester(df_stock, symbol)
        for _, row in grp.iterrows():
            result = bt.backtest_detection(row.to_dict())
            all_results.append(result)

    if not all_results:
        return pd.DataFrame()
    df_bt = pd.DataFrame(all_results)
    df_bt.to_parquet(BACKTEST_FILE, index=False)
    print(f'Backtested {len(df_bt)} detections. Saved to {BACKTEST_FILE}')
    return df_bt

df_bt = run_full_backtest(BASE_DIR / 'ohlcv', BASE_DIR / 'all_detections.parquet')
if not df_bt.empty:
    print(df_bt[['symbol','pattern','confidence','fwd_5d_return','fwd_10d_return']].head(8).to_string())

Backtested 796 detections. Saved to ps6_data\backtest_results.parquet
        symbol            pattern  confidence  fwd_5d_return  fwd_10d_return
0  ADANIENT.NS  bearish_breakdown        1.00            NaN             NaN
1  ADANIENT.NS  bearish_breakdown        1.00            NaN             NaN
2  ADANIENT.NS         double_top        0.84         -2.671          -7.419
3  ADANIENT.NS         double_top        0.35         -2.911          -2.627
4  ADANIENT.NS  bearish_breakdown        1.00          7.038          20.701
5  ADANIENT.NS  bearish_breakdown        1.00         -0.640          10.058
6  ADANIENT.NS  bearish_breakdown        1.00          0.167         -13.442
7  ADANIENT.NS         double_top        0.94         -5.531          -5.373


In [3]:
# Step 2: Win rate summary table

def pattern_win_rate_summary(df_bt: pd.DataFrame) -> pd.DataFrame:
    """Aggregate win rates by pattern type across all stocks."""
    if df_bt.empty:
        return pd.DataFrame()
    rows = []
    for pattern, grp in df_bt.groupby('pattern'):
        stats = PatternBacktester.compute_win_stats(grp.to_dict('records'))
        stats['pattern'] = pattern
        stats['count'] = len(grp)
        rows.append(stats)
    df_summary = pd.DataFrame(rows).set_index('pattern')
    return df_summary

if not df_bt.empty:
    summary = pattern_win_rate_summary(df_bt)
    print('Pattern Win Rate Summary:')
    print(summary.to_string())

Pattern Win Rate Summary:
                    5d_win_rate  5d_avg_gain  5d_avg_loss  5d_profit_factor  5d_sample_size  10d_win_rate  10d_avg_gain  10d_avg_loss  10d_profit_factor  10d_sample_size  20d_win_rate  20d_avg_gain  20d_avg_loss  20d_profit_factor  20d_sample_size  count
pattern                                                                                                                                                                                                                                                       
bearish_breakdown          53.4         2.71        -2.66              1.17             251          54.3          4.07         -4.32               1.12              243          59.0          5.68         -5.32               1.54              217    261
bullish_breakout           50.6         2.13        -2.10              1.04             267          50.9          3.33         -3.01               1.15              267          57.3          4.91         -3.

In [4]:
# Step 3: Claude API Signal Generator

class ClaudeSignalGenerator:
    """
    Converts raw pattern detections + backtest stats into
    plain-English trading signals using Claude claude-sonnet-4-20250514.

    Uses a persistent JSON cache to avoid re-calling the API
    for the same pattern+stats combination.
    """

    SYSTEM_PROMPT = """You are an expert Indian stock market analyst. 
You receive structured data about a detected chart pattern and its historical performance.
Respond with a concise, actionable signal in this exact JSON format:
{
  "headline": "One sentence summary (max 15 words)",
  "signal": "BULLISH" or "BEARISH" or "NEUTRAL",
  "explanation": "2-3 sentences explaining the pattern and what it means for the stock",
  "entry_note": "Specific entry condition to watch for",
  "stop_loss_note": "Where to place stop loss",
  "risk_reward": "Estimated risk:reward ratio",
  "confidence": "HIGH" or "MEDIUM" or "LOW"
}
Use Indian market context. Reference NSE/BSE. Mention rupee prices where relevant.
Respond ONLY with the JSON object. No preamble."""

    def __init__(self, cache_file: Path = CACHE_FILE):
        self.cache_file = cache_file
        self.cache = self._load_cache()

    def _load_cache(self) -> dict:
        if self.cache_file.exists():
            with open(self.cache_file) as f:
                return json.load(f)
        return {}

    def _save_cache(self):
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f, indent=2, default=str)

    def _cache_key(self, detection: dict, win_stats: dict) -> str:
        """Stable hash of pattern + win stats for cache lookup."""
        key_data = {
            'pattern': detection.get('pattern'),
            'symbol': detection.get('symbol'),
            'win_rate_5d': win_stats.get('5d_win_rate'),
            'win_rate_10d': win_stats.get('10d_win_rate'),
        }
        return hashlib.md5(json.dumps(key_data, sort_keys=True).encode()).hexdigest()

    def build_prompt(self, detection: dict, win_stats: dict,
                     current_price: float, sr_levels: list) -> str:
        """Build the user prompt from structured data."""
        sr_text = ', '.join([f"{s['type']} at {s['level']:.0f} ({s['distance_pct']:.1f}% away)"
                              for s in sr_levels[:4]])
        return f"""Stock: {detection.get('symbol', 'Unknown')} on NSE
Current Price: Rs {current_price:.2f}
Pattern Detected: {detection.get('pattern', '').replace('_', ' ').title()}
Pattern Confidence: {detection.get('confidence', 0):.0%}
Pattern Date: {str(detection.get('end_date', ''))[:10]}

Historical Performance of this pattern on this stock:
- 5-day win rate: {win_stats.get('5d_win_rate', 'N/A')}% (avg gain: {win_stats.get('5d_avg_gain', 'N/A')}%, avg loss: {win_stats.get('5d_avg_loss', 'N/A')}%)
- 10-day win rate: {win_stats.get('10d_win_rate', 'N/A')}% (avg gain: {win_stats.get('10d_avg_gain', 'N/A')}%, profit factor: {win_stats.get('10d_profit_factor', 'N/A')})
- 20-day win rate: {win_stats.get('20d_win_rate', 'N/A')}%
- Sample size: {win_stats.get('10d_sample_size', 0)} historical occurrences

Key Support/Resistance Levels: {sr_text or 'Not available'}
Pattern is {'bearish' if detection.get('is_bearish') else 'bullish'} by nature.

Generate the trading signal JSON now."""

    def generate_signal(self, detection: dict, win_stats: dict,
                         current_price: float, sr_levels: list = []) -> dict:
        """Generate Claude signal, using cache if available."""
        cache_key = self._cache_key(detection, win_stats)
        if cache_key in self.cache:
            return self.cache[cache_key]

        prompt = self.build_prompt(detection, win_stats, current_price, sr_levels)

        try:
            response = client.messages.create(
                model='claude-sonnet-4-20250514',
                max_tokens=600,
                system=self.SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': prompt}]
            )
            raw = response.content[0].text.strip()
            # Strip markdown fences if present
            raw = raw.replace('```json', '').replace('```', '').strip()
            signal = json.loads(raw)
            signal['cached'] = False
            self.cache[cache_key] = signal
            self._save_cache()
            time.sleep(0.5)  # rate limit courtesy
            return signal
        except Exception as e:
            fallback = {
                'headline': f'{detection.get("pattern","Pattern")} detected on {detection.get("symbol","stock")}',
                'signal': 'BEARISH' if detection.get('is_bearish') else 'BULLISH',
                'explanation': f'Pattern detected with {detection.get("confidence",0):.0%} confidence.',
                'entry_note': 'Monitor for confirmation.',
                'stop_loss_note': 'Place stop below recent low.',
                'risk_reward': '1:2',
                'confidence': 'LOW',
                'error': str(e)
            }
            return fallback

print('ClaudeSignalGenerator ready.')

ClaudeSignalGenerator ready.


In [5]:
# Step 4: Demo - generate signal for one detection
# Set ANTHROPIC_API_KEY env var before running

signal_gen = ClaudeSignalGenerator()

# Create a sample detection for demo purposes
sample_detection = {
    'symbol': 'RELIANCE.NS',
    'pattern': 'bullish_breakout',
    'confidence': 0.78,
    'is_bearish': False,
    'breakout_level': 1420.0,
    'volume_ratio': 2.3,
    'end_date': pd.Timestamp.now()
}
sample_win_stats = {
    '5d_win_rate': 72.4, '5d_avg_gain': 3.1, '5d_avg_loss': -1.8,
    '10d_win_rate': 68.0, '10d_avg_gain': 5.2, '10d_profit_factor': 1.9,
    '20d_win_rate': 65.5, '10d_sample_size': 47
}
sample_sr = [
    {'type': 'resistance', 'level': 1450.0, 'distance_pct': 2.1},
    {'type': 'support',    'level': 1380.0, 'distance_pct': 2.8}
]

print('Calling Claude API...')
signal = signal_gen.generate_signal(sample_detection, sample_win_stats, 1422.5, sample_sr)
print(json.dumps(signal, indent=2))

Calling Claude API...
{
  "headline": "bullish_breakout detected on RELIANCE.NS",
  "signal": "BULLISH",
  "explanation": "Pattern detected with 78% confidence.",
  "entry_note": "Monitor for confirmation.",
  "stop_loss_note": "Place stop below recent low.",
  "risk_reward": "1:2",
  "confidence": "LOW",
  "error": "Connection error."
}


In [6]:
# Step 5: Batch generate signals for top daily detections

def generate_daily_digest(df_bt: pd.DataFrame, ohlcv_dir: Path,
                           top_n: int = 10) -> list:
    """
    Pick top N most recent high-confidence detections.
    Generate Claude signals for each.
    Returns list of signal dicts ready for dashboard.
    """
    if df_bt.empty:
        return []

    sg = ClaudeSignalGenerator()

    # Filter recent, high confidence
    recent = df_bt.copy()
    if 'end_date' in recent.columns:
        recent['end_date'] = pd.to_datetime(recent['end_date'])
        cutoff = pd.Timestamp.now() - pd.Timedelta(days=30)
        recent = recent[recent['end_date'] >= cutoff]
    recent = recent.sort_values('confidence', ascending=False).head(top_n)

    daily_signals = []
    for _, row in recent.iterrows():
        symbol = row.get('symbol', '')
        safe = symbol.replace('.', '_')
        pf = ohlcv_dir / f'{safe}.parquet'
        if not pf.exists():
            continue
        df_s = pd.read_parquet(pf)
        current_price = df_s['close'].iloc[-1]

        # Compute aggregate win stats for this pattern+symbol combo
        same_pattern = df_bt[
            (df_bt['symbol'] == symbol) & (df_bt['pattern'] == row['pattern'])
        ]
        win_stats = PatternBacktester.compute_win_stats(same_pattern.to_dict('records'))

        signal = sg.generate_signal(row.to_dict(), win_stats, current_price)
        signal['symbol'] = symbol
        signal['pattern'] = row.get('pattern', '')
        signal['confidence_score'] = round(row.get('confidence', 0) * 100, 1)
        signal['current_price'] = round(current_price, 2)
        daily_signals.append(signal)
        print(f'  {symbol}: {signal["signal"]} — {signal["headline"]}')

    # Save for dashboard
    with open('ps6_data/daily_signals.json', 'w') as f:
        json.dump(daily_signals, f, indent=2, default=str)
    print(f'Daily digest saved: {len(daily_signals)} signals')
    return daily_signals

# Run only if backtest data exists
if not df_bt.empty:
    daily = generate_daily_digest(df_bt, BASE_DIR / 'ohlcv', top_n=5)
else:
    
    print('complete')

  ADANIENT.NS: BEARISH — bearish_breakdown detected on ADANIENT.NS
  ADANIENT.NS: BEARISH — bearish_breakdown detected on ADANIENT.NS
  ASIANPAINT.NS: BEARISH — bearish_breakdown detected on ASIANPAINT.NS
  ASIANPAINT.NS: BEARISH — bearish_breakdown detected on ASIANPAINT.NS
  AXISBANK.NS: BEARISH — bearish_breakdown detected on AXISBANK.NS
Daily digest saved: 5 signals
